In [3]:
# ===============================
# Imports
# ===============================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, classification_report

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier


# ===============================
# Load Data
# ===============================

df = pd.read_csv("steam_games_cleaned.csv")


# ===============================
# Basic Cleaning
# ===============================

df = df.drop(columns=[
    "review_count",
    "log_review_count",
    "review_sentiment",
    "developer",
    "publisher",
    "game_name"
])


# ===============================
# Genre Features
# ===============================

genre_dummies = df["genres"].str.get_dummies(",")

df = pd.concat([df.drop(columns=["genres"]), genre_dummies], axis=1)


# ===============================
# Game Age
# ===============================

current_year = 2026
df["game_age"] = current_year - df["release_year"]

df = df.drop(columns=["release_year"])


# ===============================
# Release Season
# ===============================

def get_season(m):
    if m in [12,1,2]:
        return "winter"
    elif m in [3,4,5]:
        return "spring"
    elif m in [6,7,8]:
        return "summer"
    else:
        return "fall"

df["release_season"] = df["release_month"].apply(get_season)

df = df.drop(columns=["release_month"])


# ===============================
# Price Features
# ===============================

df["log_price"] = np.log1p(df["price"])

df["price_bucket"] = pd.cut(
    df["price"],
    bins=[-1,0,500,1500,3000,df["price"].max()+1],
    labels=["Free","Cheap","Mid","Expensive","Premium"]
)

df = df.drop(columns=["price"])


# ===============================
# Score Buckets
# ===============================

df["score_bucket"] = pd.cut(
    df["review_score"],
    bins=[0,60,75,85,100],
    labels=["Bad","Average","Good","Excellent"]
)


# ===============================
# Encode Target
# ===============================

tier_map = {
    "Low":0,
    "Medium":1,
    "High":2,
    "Viral":3
}

df["popularity_tier"] = df["popularity_tier"].map(tier_map)


# ===============================
# Split Data
# ===============================

X = df.drop(columns=["popularity_tier"])
y = df["popularity_tier"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# ===============================
# Feature Groups
# ===============================

cat_cols = ["price_bucket","release_season","score_bucket"]

num_cols = [c for c in X.columns if c not in cat_cols]


# ===============================
# Preprocessing Pipeline
# ===============================

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols)
    ]
)


# ===============================
# Models
# ===============================

models = {
    "Random Forest":
        RandomForestClassifier(
            n_estimators=500,
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingClassifier(
            n_estimators=400,
            learning_rate=0.05
        ),

    "XGBoost":
        XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            eval_metric="mlogloss"
        )
}


# ===============================
# Train + Evaluate
# ===============================

for name, model in models.items():

    pipe = Pipeline([
        ("prep", preprocess),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    preds = pipe.predict(X_test)

    acc = accuracy_score(y_test, preds)

    print(f"\n{name} Accuracy: {acc:.3f}")
    print(classification_report(y_test, preds))


Random Forest Accuracy: 0.463
              precision    recall  f1-score   support

           0       0.55      0.63      0.59        98
           1       0.38      0.38      0.38        98
           2       0.34      0.24      0.28        99
           3       0.52      0.60      0.56        98

    accuracy                           0.46       393
   macro avg       0.45      0.46      0.45       393
weighted avg       0.45      0.46      0.45       393


Gradient Boosting Accuracy: 0.471
              precision    recall  f1-score   support

           0       0.59      0.64      0.62        98
           1       0.40      0.37      0.38        98
           2       0.31      0.25      0.28        99
           3       0.52      0.62      0.57        98

    accuracy                           0.47       393
   macro avg       0.46      0.47      0.46       393
weighted avg       0.46      0.47      0.46       393


XGBoost Accuracy: 0.473
              precision    recall  f1-s